# Gibbs Sampler Bayesiano per il Modello Plackett-Luce Finito (Dati Reali MovieLens)

Questo notebook replica il Gibbs sampler vettorizzato di **Caron, Teh & Murphy (Sec. 3.1)** applicandolo alle liste di preferenza reali Top-10 degli utenti MovieLens (`preference_lists_top10.csv`) arricchite con i metadati dei film (`catalogue.csv`).

Ogni sezione è divisa in:
1. **Spiegazione concettuale e sintassi Python**: cosa fa il blocco e come leggere i costrutti Python/NumPy utilizzati.
2. **Codice eseguibile**: pronto da lanciare cella per cella.

---
## 1. Importazione delle librerie

In Python, a differenza di R dove molte funzioni statistiche e grafiche sono già disponibili all'avvio, bisogna importare esplicitamente i pacchetti necessari tramite `import`:
* **`ast`**: modulo standard per convertire in sicurezza stringhe che rappresentano liste Python (es. `"[1172, 1953, ...]"` $
ightarrow$ vera lista `[1172, 1953, ...]`).
* **`csv` e `time`**: per scrivere file CSV riga per riga e misurare il tempo di esecuzione.
* **`pathlib.Path`**: gestione moderna e multipiattaforma (Windows/Mac/Linux) dei percorsi di file e cartelle, evitando problemi con `/` e `\`.
* **`numpy as np`**: libreria fondamentale per il calcolo vettoriale e matriciale (equivalente alle matrici di MATLAB o ai vettori/matrici base di R).
* **`pandas as pd`**: gestione di tabelle eterogenee (simile ai `data.frame` di R).
* **`matplotlib.pyplot as plt`**: libreria per creare grafici. Il comando `%matplotlib inline` fa sì che i grafici vengano mostrati direttamente sotto ogni cella del notebook.

In [1]:
import ast
import csv
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline

---
## 2. Parametri del modello e gestione dei percorsi

Qui impostiamo:
1. **Iperparametri della prior** ($w_k \sim \text{Gamma}(\alpha/M, \tau)$):
   * `alpha_prior = 10.0`: massa totale della prior (leggermente regolarizzata rispetto a `1.0`, migliora la stabilità numerica e la log-likelihood su film rari/non visti senza alterare la classifica).
   * `tau_prior = 1.0`: parametro di *rate* (non scala!) della Gamma; si semplifica esattamente quando normalizziamo $p_k = w_k / \sum_j w_j$.
2. **Impostazioni MCMC**: 4 catene indipendenti da 6.000 iterazioni ciascuna, scartando le prime 2.000 come *burn-in* (rimangono $4 \times 4000 = 16.000$ campioni utili).
3. **Percorsi (`PROJECT_ROOT`)**: in un notebook non esiste la variabile `__file__` tipica degli script `.py`. La piccola funzione `find_project_root()` risale automaticamente le cartelle a partire dalla directory corrente (`Path.cwd()`) finché trova `data/processed/preference_lists_top10.csv`, funzionando sia se apri il notebook dalla root del repository sia da `experiments/gibbs_sampler/`.

In [2]:
# Iperparametri e impostazioni MCMC
alpha_prior = 10.0
tau_prior = 1.0  # Gamma rate, NON scale

n_chains = 4
n_iters = 6000
burn_in = 2000
n_predictive_replicates = 200

# Quanti film mostrare nei grafici per mantenerli leggibili
top_k_forest = 25
top_k_trace = 6
top_k_acf = 8


def find_project_root() -> Path:
    """Risale le cartelle finché trova 'data/processed/preference_lists_top10.csv'."""
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "processed" / "preference_lists_top10.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Impossibile trovare 'data/processed/preference_lists_top10.csv'. "
        "Verifica di aver aperto il notebook dentro la cartella del repository."
    )


PROJECT_ROOT = find_project_root()
data_dir = PROJECT_ROOT / "data" / "processed"
preferences_path = data_dir / "preference_lists_top10.csv"
catalogue_path = data_dir / "catalogue.csv"

output_dir = PROJECT_ROOT / "results" / "gibbs_sampler" / "real"
output_dir.mkdir(parents=True, exist_ok=True)  # Crea la cartella se non esiste

print("Project root :", PROJECT_ROOT)
print("Output folder:", output_dir)

FileNotFoundError: Impossibile trovare 'data/processed/preference_lists_top10.csv'. Verifica di aver aperto il notebook dentro la cartella del repository.

---
## 3. Caricamento dei dati e rimappatura degli ID dei film

Questo passaggio risolve due differenze chiave tra dati simulati e dati reali:
1. **Parsing delle stringhe**: nel CSV `preference_lists_top10.csv`, la colonna `top_movies` contiene stringhe del tipo `"[1172, 1953, ...]"`. Il metodo `.apply(ast.literal_eval)` trasforma ogni stringa in una vera lista di interi Python.
2. **Rimappatura degli indici (`movieId` $\rightarrow$ `0 .. M-1`)**:
   * Nel dataset ci sono $M = 2095$ film distinti, ma i loro `movieId` originali hanno buchi (vanno da `1` a `163949`).
   * NumPy usa indicizzazione **0-based** (da `0` a `M-1`) per accedere agli elementi di un array. Costruiamo quindi:
     * `unique_movie_ids`: array ordinato dei 2095 `movieId` reali (serve per tornare agli ID originali a fine analisi).
     * `id_to_idx`: dizionario `{movieId: indice_0_based}` per convertire velocemente la matrice `rankings` in forma $671 \times 10$ contenente solo indici in `0..2094`.

In [ ]:
# 1. Caricamento delle liste di preferenza
prefs_df = pd.read_csv(preferences_path)
parsed_lists = prefs_df["top_movies"].apply(ast.literal_eval).tolist()

# 2. Estrazione degli ID unici e creazione della mappa movieId -> 0..M-1
unique_movie_ids = np.array(
    sorted({int(mid) for row in parsed_lists for mid in row}),
    dtype=np.int64,
)
id_to_idx = {int(mid): idx for idx, mid in enumerate(unique_movie_ids)}

# 3. Conversione delle liste in una matrice NumPy (L x m) di indici 0..M-1
rankings = np.array(
    [[id_to_idx[int(mid)] for mid in row] for row in parsed_lists],
    dtype=np.int64,
)

L, m = rankings.shape
M = len(unique_movie_ids)
print(f"Matrice rankings caricata: shape = {rankings.shape} (L={L} utenti, top-m={m}, M={M} film unici)")
print("Prime 3 righe (indici rimappati 0..2094):\n", rankings[:3])

---
## 4. Associazione dei titoli e metadati da `catalogue.csv`

Per leggere facilmente classifiche e grafici senza dover cercare a mano i `movieId`, carichiamo `catalogue.csv` e associamo a ciascuno dei `M = 2095` film:
* `titles`: titolo completo;
* `years`: anno di uscita;
* `genres_list`: generi cinematografici;
* `short_labels`: etichetta compatta tipo `"The Shawshank Redemption (1994)"` (troncata a 30 caratteri se troppo lunga) per gli assi dei grafici.

*Nota*: 15 `movieId` presenti nelle preferenze non compaiono in `catalogue.csv`; il blocco `if mid_int in cat_indexed.index: ... else: ...` li gestisce automaticamente assegnando il nome di fallback `"Movie <id>"`.

In [ ]:
cat_df = pd.read_csv(catalogue_path).drop_duplicates(subset=["movieId"])
cat_indexed = cat_df.set_index("movieId")

titles = []
years = []
genres_list = []
short_labels = []

for mid in unique_movie_ids:
    mid_int = int(mid)
    if mid_int in cat_indexed.index:
        row = cat_indexed.loc[mid_int]
        title = str(row["title"])
        year = str(int(row["year"])) if pd.notna(row["year"]) else "N/A"
        genres = str(row["genres"]) if pd.notna(row["genres"]) else ""
    else:
        title = f"Movie {mid_int}"
        year = "N/A"
        genres = ""

    titles.append(title)
    years.append(year)
    genres_list.append(genres)

    clean_title = title if len(title) <= 30 else title[:27] + "..."
    short_labels.append(f"{clean_title} ({year})" if year != "N/A" else clean_title)

print("Esempio prime 5 etichette:", short_labels[:5])

---
## 5. Il Gibbs Sampler Vettorizzato (Cuore dell'algoritmo)

Questa è la parte più importante. Ad ogni iterazione `it` della catena MCMC, il Gibbs sampler alterna due passi esatti grazie alle variabili latenti esponenziali $Z_{\ell i}$ (Caron, Teh & Murphy, Sec. 3.1):

### Passo A: Campionamento delle variabili latenti $Z_{\ell i} \mid w, \text{rankings}$
Per ogni utente $\ell \in \{0, \dots, L-1\}$ e ogni posizione $i \in \{0, \dots, m-1\}$:
$$Z_{\ell i} \sim \text{Exp}\!\left(\sum_{k=0}^{M-1} w_k - \sum_{j < i} w_{r_{\ell j}}\right)$$
Invece di usare due cicli `for` annidati su 671 utenti $\times$ 10 posizioni, usiamo l'**indicizzazione vettoriale** di NumPy:
* `selected_w = current_w[rankings]`: crea istantaneamente una matrice $671 \times 10$ sostituendo ogni indice di film con il suo peso attuale $w_k$.
* `np.cumsum(selected_w[:, :-1], axis=1)`: calcola la somma cumulativa lungo le righe (`axis=1`) dei pesi già selezionati prima dello stadio $i$.
* `rates_Z = total_w - previous_w`: ottiene in un'unica operazione matriciale tutti i $671 \times 10$ parametri di rate.
* Il blocco `if np.any(fragile):` è una guardia numerica: se per sottrazione floating-point un rate diventasse quasi zero, lo ricalcola in log-spazio con `np.logaddexp.reduce`.

### Passo B: Campionamento dei pesi $w_k \mid Z, \text{rankings}$
Per ciascun film $k \in \{0, \dots, M-1\}$:
$$w_k \sim \text{Gamma}\!\left(\frac{\alpha}{M} + n_k, \; \tau + \text{exposure}_k\right)$$
* `counts` ($n_k$): numero totale di volte in cui il film $k$ appare nelle top-10 (calcolato una sola volta prima del ciclo con `np.bincount`).
* `exposure_k`: somma di tutti gli $Z_{\ell i}$ degli stadi in cui il film $k$ era ancora disponibile. Invece di sommare su tutti gli stadi in cui $k$ è presente, usiamo il trucco complementare: prendiamo la somma totale `Z.sum()` e sottraiamo `excluded_Z`, cioè gli $Z_{\ell i}$ successivi al momento in cui $k$ è stato scelto (`future_Z`), aggregati per film tramite `np.bincount(..., weights=...)`.
* Infine salviamo `chain[it] = current_w / current_w.sum()`, ovvero le probabilità normalizzate $p_k$ (che sono identificabili, a differenza dei pesi non normalizzati $w_k$).

*Nota NumPy*: `rng.gamma(shape, scale=1.0 / rate)` usa la parametrizzazione **shape-scale** (non shape-rate!), per questo passiamo `scale=1.0 / rates_w`.

In [ ]:
t_start = time.perf_counter()

# Conteggio delle occorrenze n_k per ciascuno dei M=2095 film
counts = np.bincount(rankings.ravel(), minlength=M)
posterior_shapes = alpha_prior / M + counts  # Parametro shape della Gamma a posteriori (costante!)

# Tensore 3D che conterrà tutti i campioni: (4 catene, 6000 iterazioni, 2095 film)
samples = np.empty((n_chains, n_iters, M), dtype=np.float64)


def gibbs_chain(rng):
    """Esegue una singola catena Markov del Gibbs sampler per n_iters iterazioni."""
    current_w = rng.lognormal(mean=-0.7, sigma=0.7, size=M)
    chain = np.empty((n_iters, M), dtype=np.float64)
    tiny = np.finfo(np.float64).tiny

    for it in range(n_iters):
        # --- PASSO A: campionamento Z_li ---
        selected_w = current_w[rankings]  # Matrice (L, m)
        previous_w = np.concatenate(
            [np.zeros((L, 1)), np.cumsum(selected_w[:, :-1], axis=1)], axis=1
        )
        total_w = current_w.sum()
        rates_Z = total_w - previous_w

        # Guardia contro cancellazione numerica
        fragile = rates_Z <= np.finfo(np.float64).eps * total_w
        if np.any(fragile):
            for ell, i in np.argwhere(fragile):
                active = np.ones(M, dtype=bool)
                active[rankings[ell, :i]] = False
                rates_Z[ell, i] = np.exp(np.logaddexp.reduce(np.log(current_w[active])))

        Z = rng.exponential(scale=1.0 / rates_Z)

        # --- PASSO B: calcolo exposure_k e campionamento w_k ---
        future_Z = np.cumsum(Z[:, ::-1], axis=1)[:, ::-1] - Z
        excluded_Z = np.bincount(
            rankings.ravel(), weights=future_Z.ravel(), minlength=M
        )
        exposure = np.maximum(Z.sum() - excluded_Z, 0.0)

        rates_w = tau_prior + exposure
        current_w = np.maximum(
            rng.gamma(posterior_shapes, scale=1.0 / rates_w), tiny
        )

        # Salviamo le probabilità normalizzate p_k
        chain[it] = current_w / current_w.sum()

    return chain


# Esecuzione delle 4 catene con semi indipendenti riproducibili
for c, seed in enumerate(np.random.SeedSequence(1234).spawn(n_chains)):
    samples[c] = gibbs_chain(np.random.default_rng(seed))
    print(f"Catena {c + 1}/{n_chains} completata ({time.perf_counter() - t_start:.1f}s)")

---
## 6. Rimozione del Burn-in, Diagnostiche di Convergenza ed Esportazione

Una volta raccolti i campioni:
1. **Slicing del burn-in**: `post = samples[:, burn_in:, :]` seleziona tutte le catene (`:`), le iterazioni da `2000` alla fine (`burn_in:`), e tutti i film (`:`).
2. **Pooling**: `pooled = post.reshape(-1, M)` appiattisce le 4 catene $\times$ 4000 iterazioni in un'unica matrice $16000 \times 2095$ da cui calcoliamo media a posteriori (`p_mean`) e quantili al 2.5% e 97.5% (`p_low`, `p_high`).
3. **Diagnostiche MCMC**:
   * `split_rhat(post)`: confronta la varianza tra le catene (divise a metà) e la varianza interna a ciascuna catena. Valori $< 1.01$ indicano convergenza alla stessa distribuzione stazionaria.
   * `approximate_ess(post)`: stima l'*Effective Sample Size* (numero equivalente di campioni indipendenti) usando la regola delle coppie positive iniziali sull'autocorrelazione.
4. **Salvataggio**: esporta la tabella completa ordinata in `diagnostics.csv` e i campioni in `posterior_samples.npz`.

In [ ]:
post = samples[:, burn_in:, :]
pooled = post.reshape(-1, M)
p_mean = pooled.mean(axis=0)
p_low, p_high = np.quantile(pooled, [0.025, 0.975], axis=0)


def split_rhat(chains):
    """Split R-hat classico calcolato sui pesi normalizzati post-burn-in."""
    n = chains.shape[1] // 2
    halves = np.concatenate([chains[:, :n, :], chains[:, -n:, :]], axis=0)
    W = halves.var(axis=1, ddof=1).mean(axis=0)
    B = n * halves.mean(axis=1).var(axis=0, ddof=1)
    var_hat = ((n - 1) / n) * W + B / n
    return np.sqrt(var_hat / W)


def autocorr(x, max_lag):
    """Funzione di autocorrelazione empirica fino a max_lag."""
    x = np.asarray(x, dtype=np.float64)
    x = x - x.mean()
    denom = np.dot(x, x)
    acf = np.ones(max_lag + 1)
    if denom <= 0:
        acf[1:] = 0.0
        return acf
    for lag in range(1, max_lag + 1):
        acf[lag] = np.dot(x[:-lag], x[lag:]) / denom
    return acf


def approximate_ess(chains, max_lag=200):
    """Effective Sample Size approssimato (initial-positive-pairs rule)."""
    c, n, k = chains.shape
    limit = min(max_lag, n - 1)
    ess = np.empty(k)
    for item in range(k):
        rho = np.mean([autocorr(chains[j, :, item], limit) for j in range(c)], axis=0)
        rho_sum = 0.0
        for lag in range(1, limit, 2):
            pair = rho[lag] + rho[lag + 1]
            if pair <= 0:
                break
            rho_sum += pair
        tau_int = max(1.0, 1.0 + 2.0 * rho_sum)
        ess[item] = min(c * n, c * n / tau_int)
    return ess


rhat = split_rhat(post)
ess = approximate_ess(post)

# np.argsort(p_mean)[::-1] restituisce gli indici dei film ordinati dal più alto al più basso
rank_order = np.argsort(p_mean)[::-1]

print(f"Max split R-hat su tutti i {M} film: {rhat.max():.5f} (target < 1.01)")
print(f"Min ESS approssimato su tutti i {M} film: {ess.min():.0f} (mediana: {np.median(ess):.0f})")

# Salvataggio CSV completo
with (output_dir / "diagnostics.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(
        [
            "rank", "movieId", "title", "year", "genres",
            "observed_count", "observed_freq",
            "posterior_mean", "ci_2_5", "ci_97_5",
            "split_rhat", "ess_approx",
        ]
    )
    for rank, k in enumerate(rank_order, start=1):
        writer.writerow(
            [
                rank, int(unique_movie_ids[k]), titles[k], years[k], genres_list[k],
                int(counts[k]), counts[k] / L,
                p_mean[k], p_low[k], p_high[k], rhat[k], ess[k],
            ]
        )

# Salvataggio campioni MCMC compressi
np.savez_compressed(
    output_dir / "posterior_samples.npz",
    post_burnin_samples=post.astype(np.float32),
    unique_movie_ids=unique_movie_ids,
    titles=np.array(titles, dtype=object),
    rankings=rankings,
    burn_in=burn_in,
    alpha_prior=alpha_prior,
    tau_prior=tau_prior,
)

# Visualizzazione comoda dei Top-15 direttamente nel notebook
diag_df = pd.read_csv(output_dir / "diagnostics.csv")
diag_df.head(15)

---
## 7. Visualizzazione dei risultati

Generiamo e mostriamo direttamente nel notebook 4 grafici diagnostici (salvandoli anche nella cartella `results/gibbs_sampler/real/`):

1. **Trace Plots (Top-6)**: mostra l'andamento delle 4 catene lungo le 6.000 iterazioni. La linea verticale tratteggiata segna la fine del burn-in (`2000`).
2. **Forest Plot (Top-25)**: media a posteriori e intervallo di credibilità al 95% per i primi 25 film.
3. **Autocorrelazione post-burn-in (Top-8)**: verifica quanto rapidamente scende la correlazione tra iterazioni successive.
4. **Posterior Predictive Check**: simula 200 dataset sintetici estraendo top-10 dai campioni a posteriori tramite il trucco di **Gumbel-top-k** (`log(p) + Gumbel(0,1)`, matematicamente equivalente all'estrazione sequenziale Plackett-Luce) e confronta le frequenze simulate con quelle osservate nei dati reali.

In [ ]:
# ------------------------------------------------------------
# 7.1 Trace plots per i Top-6 film
# ------------------------------------------------------------
top_trace_idx = rank_order[:top_k_trace]
fig, axes = plt.subplots(3, 2, figsize=(14, 10), sharex=True)

for ax, k in zip(axes.ravel(), top_trace_idx):
    for c in range(n_chains):
        ax.plot(samples[c, :, k], alpha=0.6, linewidth=0.6, label=f"Chain {c + 1}")
    ax.axvline(burn_in, color="black", linestyle="--", linewidth=1.0, label="Burn-in")
    ax.axhline(p_mean[k], color="crimson", linestyle=":", linewidth=1.2, label="Post. mean")
    ax.set_title(short_labels[k], fontsize=10, fontweight="bold")
    ax.set_ylabel("p")
    ax.grid(alpha=0.2)

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.955), ncol=n_chains + 2, fontsize=8.5, frameon=True)
axes[-1, 0].set_xlabel("Iteration")
axes[-1, 1].set_xlabel("Iteration")
fig.suptitle("Trace Plots: Top-6 Movies by Posterior Mean Weight", fontsize=13, y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.92])
fig.savefig(output_dir / "01_trace_plots.png", dpi=180)
plt.show()

# ------------------------------------------------------------
# 7.2 Forest plot orizzontale per i Top-25 film
# ------------------------------------------------------------
top_forest_idx = rank_order[:top_k_forest][::-1]  # Invertito così il #1 sta in alto
y_pos = np.arange(top_k_forest)

fig, ax = plt.subplots(figsize=(11, 8.5))
ax.errorbar(
    p_mean[top_forest_idx],
    y_pos,
    xerr=[
        p_mean[top_forest_idx] - p_low[top_forest_idx],
        p_high[top_forest_idx] - p_mean[top_forest_idx],
    ],
    fmt="o",
    color="#1f77b4",
    ecolor="#1f77b4",
    elinewidth=1.4,
    capsize=3.5,
    markersize=5.5,
    label="Posterior mean + 95% credible interval",
)
ax.set_yticks(y_pos)
ax.set_yticklabels(
    [f"#{top_k_forest - i}  {short_labels[k]}" for i, k in enumerate(top_forest_idx)],
    fontsize=9,
)
ax.set_xlabel("Normalized Plackett-Luce weight (p)")
ax.set_title(f"Top-{top_k_forest} Movies: Posterior Mean & 95% Credible Intervals")
ax.grid(axis="x", alpha=0.25)
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(output_dir / "02_posterior_intervals.png", dpi=180)
plt.show()

# ------------------------------------------------------------
# 7.3 Autocorrelazione post-burn-in per i Top-8 film
# ------------------------------------------------------------
top_acf_idx = rank_order[:top_k_acf]
max_lag = min(100, post.shape[1] - 1)
cmap = plt.get_cmap("tab10")

fig, ax = plt.subplots(figsize=(11, 5.5))
for i, k in enumerate(top_acf_idx):
    rho = np.mean([autocorr(post[c, :, k], max_lag) for c in range(n_chains)], axis=0)
    ax.plot(
        np.arange(max_lag + 1),
        rho,
        linewidth=1.4,
        color=cmap(i),
        label=f"#{i + 1} {short_labels[k]}",
    )
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Lag")
ax.set_ylabel("Mean autocorrelation across chains")
ax.set_title(f"Post-Burn-In Autocorrelation (Top-{top_k_acf} Movies)")
ax.grid(alpha=0.25)
ax.legend(fontsize=8.5, ncol=2, loc="upper right")
fig.tight_layout()
fig.savefig(output_dir / "03_autocorrelation.png", dpi=180)
plt.show()

# ------------------------------------------------------------
# 7.4 Posterior Predictive Check (Top-25 + Calibrazione su M=2095)
# ------------------------------------------------------------
rng_ppc = np.random.default_rng(2026)
rep_indices = rng_ppc.choice(len(pooled), size=n_predictive_replicates, replace=True)
ppc_rates = np.empty((n_predictive_replicates, M), dtype=np.float64)

for r, idx in enumerate(rep_indices):
    log_p = np.log(pooled[idx])
    scores = log_p[None, :] + rng_ppc.gumbel(size=(L, M))
    top_m_sim = np.argpartition(scores, kth=M - m, axis=1)[:, -m:]
    ppc_rates[r] = np.bincount(top_m_sim.ravel(), minlength=M) / L

observed_rate = counts / L
ppc_mean = ppc_rates.mean(axis=0)
ppc_low, ppc_high = np.quantile(ppc_rates, [0.025, 0.975], axis=0)

fig, (ax_left, ax_right) = plt.subplots(
    1, 2, figsize=(15, 8.5), gridspec_kw={"width_ratios": [1.45, 1.0]}
)

ax_left.errorbar(
    ppc_mean[top_forest_idx],
    y_pos,
    xerr=[
        ppc_mean[top_forest_idx] - ppc_low[top_forest_idx],
        ppc_high[top_forest_idx] - ppc_mean[top_forest_idx],
    ],
    fmt="o",
    color="#1f77b4",
    capsize=3.5,
    markersize=5,
    label="Posterior predictive mean + 95% interval",
)
ax_left.scatter(
    observed_rate[top_forest_idx],
    y_pos,
    marker="x",
    color="crimson",
    s=55,
    linewidths=1.6,
    zorder=4,
    label="Observed top-10 frequency",
)
ax_left.set_yticks(y_pos)
ax_left.set_yticklabels(
    [f"#{top_k_forest - i}  {short_labels[k]}" for i, k in enumerate(top_forest_idx)],
    fontsize=9,
)
ax_left.set_xlabel("Fraction of users with movie in top-10")
ax_left.set_title(f"Top-{top_k_forest} Movies: Observed vs. Predictive")
ax_left.grid(axis="x", alpha=0.25)
ax_left.legend(loc="lower right", fontsize=8.5)

ax_right.scatter(observed_rate, ppc_mean, alpha=0.35, s=16, color="#2ca02c", edgecolors="none")
max_val = max(observed_rate.max(), ppc_mean.max()) * 1.05
ax_right.plot([0, max_val], [0, max_val], "k--", linewidth=1.0, label="Ideal calibration (y = x)")
ax_right.set_xlabel("Observed top-10 inclusion frequency")
ax_right.set_ylabel("Posterior predictive mean frequency")
ax_right.set_title(f"Full-Catalog Calibration (M = {M} movies)")
ax_right.grid(alpha=0.25)
ax_right.legend(loc="upper left", fontsize=8.5)

fig.suptitle("Posterior Predictive Check: Top-10 Inclusion Frequencies", fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig(output_dir / "04_posterior_predictive.png", dpi=180)
plt.show()

---
## 8. (Bonus) Validazione Out-of-Sample con 5-Fold Cross-Validation

Questa cella opzionale misura quanto bene il modello generalizza su **utenti mai visti** dividendo i 671 utenti in 5 fold (80% train, 20% test) e calcolando la **log-likelihood predittiva Plackett-Luce** media sul test set rispetto a due baseline semplici:
* **Uniforme**: tutti i 2095 film hanno probabilità $1/M$.
* **Frequenza empirica**: probabilità proporzionale ai soli conteggi nel training set (senza considerare la posizione 1..10).

In [ ]:
def fit_pl_posterior_mean(train_rankings, alpha=10.0, tau=1.0, n_iters_cv=2000, burn_in_cv=600, seed=42):
    rng = np.random.default_rng(seed)
    L_tr = len(train_rankings)
    counts_tr = np.bincount(train_rankings.ravel(), minlength=M)
    shapes = alpha / M + counts_tr
    current_w = rng.lognormal(mean=-0.7, sigma=0.7, size=M)
    p_sum = np.zeros(M, dtype=np.float64)
    tiny = np.finfo(np.float64).tiny

    for it in range(n_iters_cv):
        selected_w = current_w[train_rankings]
        prev_w = np.concatenate([np.zeros((L_tr, 1)), np.cumsum(selected_w[:, :-1], axis=1)], axis=1)
        rates_Z = np.maximum(current_w.sum() - prev_w, tiny)
        Z = rng.exponential(scale=1.0 / rates_Z)
        future_Z = np.cumsum(Z[:, ::-1], axis=1)[:, ::-1] - Z
        excluded_Z = np.bincount(train_rankings.ravel(), weights=future_Z.ravel(), minlength=M)
        exposure = np.maximum(Z.sum() - excluded_Z, 0.0)
        current_w = np.maximum(rng.gamma(shapes, scale=1.0 / (tau + exposure)), tiny)
        if it >= burn_in_cv:
            p_sum += current_w / current_w.sum()

    return p_sum / (n_iters_cv - burn_in_cv)


def mean_pl_loglik(test_rankings, p):
    """Log-likelihood Plackett-Luce media per utente sul test set."""
    selected_p = p[test_rankings]
    removed_p = np.concatenate([np.zeros((len(test_rankings), 1)), np.cumsum(selected_p[:, :-1], axis=1)], axis=1)
    denom = np.maximum(1.0 - removed_p, 1e-15)
    return float(np.mean(np.sum(np.log(selected_p / denom), axis=1)))


rng_cv = np.random.default_rng(2026)
perm = rng_cv.permutation(L)
folds = np.array_split(perm, 5)

scores_pl, scores_freq, scores_unif = [], [], []
p_unif = np.full(M, 1.0 / M)

for fold_idx, test_idx in enumerate(folds):
    train_idx = np.setdiff1d(perm, test_idx)
    tr, te = rankings[train_idx], rankings[test_idx]

    p_pl = fit_pl_posterior_mean(tr, alpha=alpha_prior, seed=100 + fold_idx)
    counts_tr = np.bincount(tr.ravel(), minlength=M) + (alpha_prior / M)
    p_freq = counts_tr / counts_tr.sum()

    scores_pl.append(mean_pl_loglik(te, p_pl))
    scores_freq.append(mean_pl_loglik(te, p_freq))
    scores_unif.append(mean_pl_loglik(te, p_unif))

print("5-Fold Mean Test PL Log-Likelihood (più alto = migliore):")
print(f"  Uniform baseline   : {np.mean(scores_unif):.3f} +/- {np.std(scores_unif):.3f}")
print(f"  Frequency smoothed : {np.mean(scores_freq):.3f} +/- {np.std(scores_freq):.3f}")
print(f"  Bayesian PL (Gibbs): {np.mean(scores_pl):.3f} +/- {np.std(scores_pl):.3f}")